1. Lấy comment các video từ Youtube, dùng Youtube Data API V3

In [2]:
import importlib
from src import youtube_crawler

In [5]:
importlib.reload(youtube_crawler)

all_comments = youtube_crawler.get_youtube_comments()

Tổng số comment lấy được: 19463
sao giờ nghe bài này hay vậy ta
xin lỗi anh Quí vì thời đó đã chê anh
Mặc kệ chê t thấy hay vc
đừng đùa với lượt view của anh ấy
sao lai lam dc the


2. Cào comment từ VOZ

In [1]:
!pip install cloudscraper beautifulsoup4 ddgs pandas selenium webdriver-manager

   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ---------------------------------------- 3.1/3.1 MB 23.1 MB/s  0:00:00

   ------------------------------ --------- 3/4 [cloudscraper]
   ---------------------------------------- 4/4 [cloudscraper]



In [2]:
!pip install undetected-chromedriver

In [3]:
import time
import pandas as pd
import warnings
import unicodedata
import os
import re
import random
import subprocess
from bs4 import BeautifulSoup
from ddgs import DDGS
import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, WebDriverException

# --- CAU HINH DANH SACH TU KHOA ---
KEYWORDS = [
    # "BKAV",
    # "Bóc phốt",
    # "Lương lập trình viên",
    # "Thất nghiệp",
    # "Bắc kỳ",
    # "Bake",
    # "TNT",
    # "Rồ Nga",
    # "Cổ nâu",
    # "Mê Tây",
    # "Cơm sườn",
    # "Vin",
    # "Vin nô",
    # "Táo",
    # "Lùa gà",
    # "Úp bô",
    # "Hold to die",
    # "Chuyên gia đọc lệnh",
    # "Tự nhục",
    # "Loser",
    # "Winner",
    # "Simp",
    # "Nữ quyền",
    # "Beta cuckold",
    # "Chui chạn",
    # "Đổ vỏ",
    # Vùng miền & Phân biệt (Slang)
    "Thanh lịch",          # Ám chỉ người Hà Nội (mỉa mai)
    "Phóng khoáng",        # Ám chỉ người Sài Gòn/Miền Nam (mỉa mai)
    "Vương quốc",          # Ám chỉ Nghệ An/Hà Tĩnh
    "Hoa thanh quế",       # Ám chỉ Thanh Hóa
    "Nước ngoài",          # Slang gọi lái vùng 4 (TNT)
    "Trại súc vật",        # Từ ngữ cực đoan ám chỉ xã hội

    # Chính trị & Phe phái (Slang)
    "Bò đỏ",               # Dư luận viên/Phe ủng hộ cực đoan
    "Bò vàng",             # Phe chống đối/3 que
    "Lực lượng 47",        # Dư luận viên
    "Orc",                 # Ám chỉ Nga/Người ủng hộ Nga
    "Rồ Mỹ",               # Người cuồng Mỹ
    "Rồ Trung",            # Người cuồng TQ
    "Tư bản",              # Thường dùng trong các context so sánh chế độ

    # Giới tính & Đời sống (Toxic)
    "Dẩm",                 # Chỉ phụ nữ/đàn ông tính khí thất thường, ảo tưởng
    "Gái dẩm",             # (Cụ thể hơn)
    "Khọm già",            # Mạt sát phụ nữ lớn tuổi
    "Single mom",          # Đối tượng thường bị công kích dữ dội trên Voz
    "Nuôi tu hú",          # Ám chỉ việc nuôi con người khác (tương tự đổ vỏ)
    "Gạ gẫm",              # Các post bóc phốt quấy rối
    "Sugar baby",          # Chủ đề bao nuôi
    "Pê đê",               # Từ ngữ kỳ thị LGBT (thường gặp ở F17)

    # Xã hội & Định kiến
    "Bần nông",            # Chửi người khác quê mùa, tư duy thấp kém
    "Thượng đẳng",         # Chửi người khác tỏ vẻ cao sang
    "Tiêu chuẩn kép",      # Double standard (tranh cãi gay gắt)
    "Thợ dạy",             # Mạt sát giáo viên
    "Xăm trổ",             # Định kiến người xăm mình
    "Thu tha",             # Cờ bạc, bóng bánh, nợ nần
    "Báo nhà",             # Con cái phá gia chi tử
    "Gen Z",               # Cuộc chiến thế hệ (chê Gen Z lười, yếu đuối)
    "Nằm thẳng",           # Trào lưu không phấn đấu (gây tranh cãi với phe nỗ lực)

    # Giao thông & Ý thức
    "Ninja Lead",          # Phụ nữ đi xe máy
    "Bán tải",             # "Vua lấn làn", ý thức kém
    "Xe điên",             # Tai nạn giao thông
    "Trẻ trâu",            # Đua xe, quậy phá

    # Kinh tế & Lừa đảo
    "Công nghệ lõi",       # Mỉa mai buôn đất (Phân lô bán nền)
    "Phân lô bán nền",     # (Cụ thể)
    "Cò đất",              # Môi giới BĐS
    "Đa cấp",              # Chủ đề lừa đảo
    "Bảo hiểm",            # (Manulife, v.v... tranh cãi lừa đảo)

    # Cộng đồng VOZ
    "Vốt dơ",              # Danh xưng (thường dùng để tự chửi)
    "Lùn đụt cận trĩ",     # Meme miêu tả Vozer
    "Reset",
]

MAX_THREADS_PER_KEYWORD = 5
MAX_PAGES = 30
TIMEOUT_LIMIT = 20
OFFSET_X = -1000 # Toa do man hinh phu

warnings.filterwarnings("ignore")

def get_output_path(keyword, threads, pages):
    text = unicodedata.normalize('NFD', keyword)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = text.replace('đ', 'd').replace('Đ', 'D')
    clean_keyword = text.replace(' ', '_')
    filename = f"{clean_keyword}_{threads}_{pages}.csv"
    output_dir = os.path.join(os.pardir, "data", "raw", "voz")
    os.makedirs(output_dir, exist_ok=True)
    return os.path.join(output_dir, filename)

def get_driver():
    print("[INIT] Khoi tao Driver moi...")

    # Duong dan profile (Hay xoa thu muc nay neu chay bi loi lien tuc)
    user_data_dir = os.path.join(os.getcwd(), "chrome_profile_fixed")

    # --- HAM CON: TAO OPTIONS MOI TINH ---
    def create_fresh_options():
        opts = uc.ChromeOptions()
        opts.add_argument("--disable-blink-features=AutomationControlled")
        opts.add_argument("--disable-dev-shm-usage")
        opts.add_argument("--no-sandbox")
        opts.page_load_strategy = 'normal'
        return opts

    driver = None

    # THU LAN 1: Voi version 143 + use_subprocess=True
    try:
        driver = uc.Chrome(
            options=create_fresh_options(),
            user_data_dir=user_data_dir,
            version_main=143,
            use_subprocess=True # [QUAN TRONG] Giup tranh loi 'chrome not reachable'
        )
    except Exception as e:
        print(f"    [WARN] Khoi tao v143 that bai ({e}). Thu lai voi version tu dong...")

        # THU LAN 2: Voi version None
        try:
            driver = uc.Chrome(
                options=create_fresh_options(),
                user_data_dir=user_data_dir,
                version_main=None,
                use_subprocess=True # [QUAN TRONG]
            )
        except Exception as e2:
            print(f"    [FATAL] Khong the khoi tao Driver: {e2}")
            # Neu van loi, hay thu xoa thu muc 'chrome_profile_fixed' va chay lai
            raise e2

    # Move sang man hinh phu (neu co)
    try:
        driver.set_window_position(OFFSET_X, 0)
        time.sleep(1)
        driver.maximize_window()
    except: pass

    driver.set_page_load_timeout(TIMEOUT_LIMIT)

    # Chan quang cao
    try:
        driver.execute_cdp_cmd('Network.enable', {})
        driver.execute_cdp_cmd('Network.setBlockedURLs', {
            "urls": ["*googleads*", "*doubleclick*", "*googlesyndication*", "*adservice*", "*google_vignette*", "*.gif"]
        })
    except: pass

    return driver

def force_kill_driver(driver):
    """Giet tien trinh Chrome manh tay"""
    pid = None
    if driver:
        try:
            pid = driver.service.process.pid
        except: pass
        try: driver.quit()
        except: pass

    # Neu driver da chet nhung chrome van con, hoac khong lay duoc PID
    # Kill toan bo chrome.exe cua user hien tai (Can than neu ban dang dung chrome khac)
    if pid:
        try:
            subprocess.call(['taskkill', '/F', '/T', '/PID', str(pid)],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"    [SYSTEM] Killed PID: {pid}")
        except: pass

def append_batch_to_csv(file_path, batch_data):
    if not batch_data: return 0
    file_exists = os.path.isfile(file_path)
    start_id = 1
    if file_exists:
        try:
            df_check = pd.read_csv(file_path, usecols=['id'])
            if not df_check.empty:
                start_id = df_check['id'].max() + 1
        except: start_id = 1

    df_new = pd.DataFrame(batch_data, columns=['text'])
    df_new.insert(0, 'id', range(start_id, start_id + len(df_new)))
    df_new.to_csv(file_path, mode='a', header=not file_exists, index=False, encoding='utf-8-sig')
    return len(df_new)

def search_voz(keyword, limit):
    print(f"[SEARCH] Tim link cho: {keyword}")
    results = []
    try:
        with DDGS() as ddgs:
            gen = ddgs.text(f"site:voz.vn {keyword}", max_results=limit + 10)
            for r in gen:
                if "/t/" in r['href'] and r['href'] not in results:
                    results.append(r['href'])
                if len(results) >= limit: break
    except Exception as e:
        print(f"[ERROR] Search loi: {e}")
    return results

def scrape_comments(driver, url, max_pages):
    current_thread_comments = []
    base_url = url.split('/page-')[0]
    if base_url.endswith('/'): base_url = base_url[:-1]

    print(f"  -> Scraping: {base_url}")

    for page in range(1, max_pages + 1):
        try:
            target_url = base_url if page == 1 else f"{base_url}/page-{page}"
            try:
                driver.get(target_url)
            except TimeoutException:
                driver.execute_script("window.stop();")

            if "Just a moment" in driver.title:
                print("    [ALERT] Cloudflare! Waiting...")
                time.sleep(5)

            try:
                driver.execute_script("""
                    var vignette = document.getElementById('google_vignette_modal');
                    if (vignette) vignette.remove();
                    document.body.style.overflow = 'auto';
                """)
            except: pass

            time.sleep(random.uniform(2, 3))

            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # [CHECK QUAN TRONG] Neu khong thay noi dung -> Trang loi -> Bao loi de restart
            posts = soup.select('.message-inner')
            if not posts:
                # Thu cho them 2s
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                posts = soup.select('.message-inner')

                if not posts:
                    print("    [WARN] Trang trong hoac bi chan (0 post).")
                    break # Thoat vong lap page

            for post in posts:
                content_tag = post.select_one('.bbWrapper')
                if content_tag:
                    for q in content_tag.find_all('blockquote'): q.decompose()
                    for br in content_tag.find_all('br'): br.replace_with('\n')

                    text = content_tag.get_text(separator='\n', strip=True)
                    text = re.sub(r'[ \t]*\n[ \t]*', '\n', text)
                    text = re.sub(r'\n+', '\n', text)

                    if text:
                        current_thread_comments.append(text)

            if not soup.select_one('a.pageNav-jump--next'): break

        except WebDriverException:
            raise
        except Exception:
            break

    return current_thread_comments

def main():
    driver = None

    try:
        driver = get_driver()

        for keyword in KEYWORDS:
            print(f"\n{'='*50}\n🚀 BAT DAU TU KHOA: {keyword}\n{'='*50}")

            output_path = get_output_path(keyword, MAX_THREADS_PER_KEYWORD, MAX_PAGES)
            urls = search_voz(keyword, MAX_THREADS_PER_KEYWORD)

            if not urls: continue

            total_saved_for_keyword = 0

            for i, url in enumerate(urls):
                print(f"\n[KEYWORD: {keyword}] [THREAD {i+1}/{len(urls)}]")

                try:
                    batch_data = scrape_comments(driver, url, MAX_PAGES)

                    if batch_data:
                        count = append_batch_to_csv(output_path, batch_data)
                        total_saved_for_keyword += count
                        print(f"    [SUCCESS] + {count} dòng.")
                    else:
                        # [FIX LOGIC] Neu khong co data -> Coi nhu trinh duyet bi hong
                        # Raise loi de nhay xuong except -> Restart Driver
                        print("    [WARN] Khong co data -> Nghi ngo trinh duyet treo.")
                        raise WebDriverException("Zero data returned (Zombie Browser)")

                except Exception as e:
                    print(f"    [CRITICAL] Phat hien loi/Treo: {e}")
                    print("    [RECOVERY] Restarting Driver...")

                    # 1. Giet chet trinh duyet cu
                    force_kill_driver(driver)

                    # 2. Khoi tao cai moi
                    try:
                        driver = get_driver()
                        print("    [RECOVERY] Driver moi san sang.")
                    except:
                        print("    [FATAL] Khong the restart driver.")
                        break
                    continue

            time.sleep(2)

    finally:
        force_kill_driver(driver)

if __name__ == "__main__":
    main()

[INIT] Khoi tao Driver moi...

🚀 BAT DAU TU KHOA: Thanh lịch
[SEARCH] Tim link cho: Thanh lịch

[KEYWORD: Thanh lịch] [THREAD 1/5]
  -> Scraping: https://voz.vn/t/Đa-nang-ghi-nhan-luong-mua-ngay-vuot-moc-lich-su-41-nam.864110
    [SUCCESS] + 20 dòng.

[KEYWORD: Thanh lịch] [THREAD 2/5]
  -> Scraping: https://voz.vn/t/elon-musk-sap-va-mom-troc-phu-hoc-đoi-du-lich-vu-tru.387640
    [WARN] Khong co data -> Nghi ngo trinh duyet treo.
    [CRITICAL] Phat hien loi/Treo: Message: Zero data returned (Zombie Browser)

    [RECOVERY] Restarting Driver...
    [SYSTEM] Killed PID: 27408
[INIT] Khoi tao Driver moi...
    [RECOVERY] Driver moi san sang.

[KEYWORD: Thanh lịch] [THREAD 3/5]
  -> Scraping: https://voz.vn/t/du-lich-nhat-ban-bung-no-du-khach-trung-quoc-“quay-lung”.1200694
    [SUCCESS] + 131 dòng.

[KEYWORD: Thanh lịch] [THREAD 4/5]
  -> Scraping: https://voz.vn/t/song-to-lich-vua-trong-xanh-lai-tai-dien-tinh-trang-o-nhiem.1143719
    [SUCCESS] + 81 dòng.

[KEYWORD: Thanh lịch] [THREAD 5